In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# 04. Generative AI: Fine-Tuning LLMs with LoRA and PEFT

## Algorithm Category
**Type**: Generative AI - Fine-Tuning  
**Complexity**: High  
**Use Case**: Parameter-efficient fine-tuning of large models

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand LoRA (Low-Rank Adaptation) and why it's needed
- Understand the difference between full fine-tuning and parameter-efficient fine-tuning
- Use PEFT (Parameter-Efficient Fine-Tuning) library for LoRA
- Configure LoRA adapters (rank, alpha, target modules)
- Train LoRA adapters on custom datasets
- Merge and save LoRA weights
- Apply LoRA adapters to base models

## Historical Context

LoRA was introduced by Microsoft Research in 2021:
- Hu, E.J., et al. (2021): "LoRA: Low-Rank Adaptation of Large Language Models"
- Revolutionized fine-tuning by reducing trainable parameters by 100-1000×
- Made fine-tuning accessible without expensive GPUs
- Foundation for many modern fine-tuning approaches

**Key Papers/References:**
- Hu, E.J., et al. (2021). "LoRA: Low-Rank Adaptation of Large Language Models"
- PEFT library: Hugging Face's implementation of various PEFT methods

## What is LoRA?

**LoRA (Low-Rank Adaptation)** is a parameter-efficient fine-tuning technique that:
- Adds small trainable matrices to model layers
- Freezes original model weights (no changes)
- Only trains the new adapter matrices
- Reduces trainable parameters dramatically (e.g., 7B model: 7B → 4M parameters)

### Why LoRA?

**Full Fine-Tuning Problems:**
- Requires training all model parameters (billions)
- Needs multiple GPUs with lots of memory
- Expensive and time-consuming
- Risk of catastrophic forgetting

**LoRA Benefits:**
- ✅ Train only 0.1-1% of parameters
- ✅ Much lower memory requirements
- ✅ Faster training
- ✅ Can train on single GPU
- ✅ Multiple LoRA adapters per model
- ✅ Easy to switch between adapters

### Key Concepts

**Low-Rank Decomposition**: Representing weight updates as product of two small matrices
- Original: ΔW (full size: d×d)
- LoRA: ΔW = BA where B (d×r) and A (r×d), r << d
- r (rank): Small dimension (typically 4-64)
- Result: Much fewer parameters to train

**Adapter Layers**: Small matrices added to model
- Inserted into attention layers (Q, K, V projections)
- Or feed-forward layers
- Trained while base model stays frozen
- Can be merged into base model after training

**PEFT Library**: Hugging Face library for parameter-efficient fine-tuning
- Supports LoRA, Prefix Tuning, P-Tuning, and more
- Easy integration with Transformers library
- Handles adapter loading/saving automatically

### When to Use LoRA

✅ **Good for:**
- Fine-tuning large models on limited hardware
- Domain-specific adaptation (medical, legal, code)
- Task-specific fine-tuning (classification, generation)
- When you want to preserve base model capabilities
- Creating multiple specialized versions of a model
- Training on small datasets

❌ **Not ideal for:**
- Very different tasks (may need full fine-tuning)
- When you have unlimited compute resources
- When maximum performance is critical (full fine-tuning may be better)
- Very small models (overhead not worth it)


## Theory & Mechanics

### What is LoRA?

LoRA enables fine-tuning with minimal parameters by using low-rank decomposition.

**Mathematical Foundation:**

Instead of updating full weight matrix W:
$$W_{new} = W_{old} + \Delta W$$

LoRA decomposes the update:
$$\Delta W = BA$$

Where:
- $B \in \mathbb{R}^{d \times r}$: Low-rank matrix (d = original dimension, r = rank)
- $A \in \mathbb{R}^{r \times d}$: Low-rank matrix
- $r << d$: Rank is much smaller than dimension (e.g., r=8, d=4096)

**Parameter Reduction:**
- Full fine-tuning: $d \times d$ parameters (e.g., 4096×4096 = 16M)
- LoRA: $d \times r + r \times d = 2rd$ parameters (e.g., 2×4096×8 = 65K)
- Reduction: ~250× fewer parameters!

### How LoRA Works

1. **Freeze Base Model**: Original weights stay unchanged
2. **Add Adapter Matrices**: Insert B and A matrices into layers
3. **Forward Pass**: $h = Wx + BAx$ (base + adapter)
4. **Backward Pass**: Only update B and A (not W)
5. **Merge (Optional)**: Combine adapter into base model after training

### PEFT Library

**PEFT (Parameter-Efficient Fine-Tuning)** provides:
- LoRA implementation
- Easy configuration (rank, alpha, target modules)
- Adapter management (save, load, merge)
- Integration with Hugging Face Transformers

**Key Configuration:**
- **r (rank)**: Dimension of low-rank matrices (4-64 typical)
- **alpha**: Scaling factor for adapter (usually = r)
- **target_modules**: Which layers to add adapters to
- **dropout**: Regularization for adapters


## Installation & Setup


## Implementation


In [ ]:
# ============================================
# SETTING UP PYTHON PATH: Accessing Project Modules
# ============================================

# Add project root to Python path for imports
# This allows us to import modules from the src/ directory
import sys  # sys: System-specific parameters and functions
from pathlib import Path  # Path: Object-oriented filesystem paths

# Get the project root (two levels up from this notebook)
# notebooks/generative_ai/ -> notebooks/ -> project_root/
project_root = Path().resolve().parent.parent
# Path().resolve(): Get current directory (notebooks/generative_ai/)
# .parent: Go up one level (notebooks/)
# .parent: Go up another level (project_root/)

# Add project root to Python path if not already there
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    # sys.path: List of directories Python searches for modules
    # .insert(0, ...): Add to beginning (highest priority)
    # This allows: from src.llm.peft_utils import ...

# ============================================
# IMPORTING PEFT UTILITIES: LoRA Fine-Tuning Functions
# ============================================

# Our custom PEFT/LoRA utilities
from src.llm.peft_utils import setup_lora_model, prepare_model_for_training, merge_lora_weights
# setup_lora_model(): Configure LoRA adapters for a model
#   - Sets up PEFT configuration (rank, alpha, target modules)
#   - Adds adapter layers to specified modules
#   - Freezes base model weights
# prepare_model_for_training(): Prepare model for training
#   - Enables gradient checkpointing (saves memory)
#   - Sets up training mode
#   - Configures optimizer and learning rate scheduler
# merge_lora_weights(): Merge adapter weights into base model
#   - Combines LoRA adapters with base weights
#   - Creates merged model (no separate adapters needed)
#   - Useful for inference (faster, no adapter overhead)

print("PEFT utilities imported")  # Confirm imports successful

# ============================================
# KEY CONCEPTS
# ============================================

# LoRA (Low-Rank Adaptation):
# 1. Adds small trainable matrices (adapters) to model layers
# 2. Freezes original model weights (no changes)
# 3. Only trains adapter matrices (much fewer parameters)
# 4. Reduces memory and compute requirements dramatically
#
# PEFT Library:
# - Provides LoRA implementation
# - Easy configuration and management
# - Integrates with Hugging Face Transformers
# - Supports multiple PEFT methods
#
# Typical Workflow:
# 1. Load base model (e.g., Llama 2, Mistral)
# 2. Setup LoRA adapters (configure rank, alpha, target modules)
# 3. Prepare for training (gradient checkpointing, etc.)
# 4. Train on custom dataset
# 5. Save adapters (small files, can share easily)
# 6. Load adapters for inference (or merge into base model)
#
# Example Configuration:
# - rank=8: Low-rank dimension (smaller = fewer parameters)
# - alpha=16: Scaling factor (usually 2× rank)
# - target_modules=["q_proj", "v_proj"]: Which layers to adapt
# - dropout=0.1: Regularization

## Validation & Testing


In [ ]:
# Validation
print("Validation")

## Performance Benchmarking


In [ ]:
# Performance
print("Performance")

## Summary & Key Takeaways

- Key concept 1
- Key concept 2
- Key concept 3
